# Dataset - ExtractMethod

In [4]:
# !pip install -r requirements.txt

In [5]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib
import matplotlib.pyplot as plt # for data visualization
%matplotlib inline
from IPython.display import Image

In [6]:
data = pd.read_csv("../datasets/Random_Generated_Dataset_150k.csv")
print("Dimensões dataset:", data.shape)

data=data.set_index("id_")

print("Tipagem dos dados:\n", data.dtypes)

Dimensões dataset: (150000, 24)
Tipagem dos dados:
 methodAnonymousClassesQty     int64
methodAssignmentsQty          int64
methodCbo                     int64
methodComparisonsQty          int64
methodLambdasQty              int64
methodLoc                     int64
methodLoopQty                 int64
methodMathOperationsQty       int64
methodMaxNestedBlocks         int64
methodNumbersQty              int64
methodParametersQty           int64
methodParenthesizedExpsQty    int64
methodReturnQty               int64
methodRfc                     int64
methodStringLiteralsQty       int64
methodSubClassesQty           int64
methodTryCatchQty             int64
methodUniqueWordsQty          int64
methodVariablesQty            int64
methodWmc                     int64
bugFixCount                   int64
refactoringsInvolved          int64
y                             int64
dtype: object


In [7]:
data.head()

,methodAnonymousClassesQty,methodAssignmentsQty,methodCbo,methodComparisonsQty,methodLambdasQty,methodLoc,methodLoopQty,methodMathOperationsQty,methodMaxNestedBlocks,methodNumbersQty,...,methodRfc,methodStringLiteralsQty,methodSubClassesQty,methodTryCatchQty,methodUniqueWordsQty,methodVariablesQty,methodWmc,bugFixCount,refactoringsInvolved,y
id_,,,,,,,,,,,,,,,,,,,,,
27308,0,0,0,0,0,3,0,0,0,0,...,0,0,0,0,4,0,1,0,0,1
19994,0,10,1,0,0,25,2,2,3,0,...,8,7,0,0,32,8,6,3,5,1
29413,0,2,5,0,0,15,0,0,2,0,...,8,1,0,1,16,1,3,1,1,1
52702,0,23,11,2,0,57,2,1,3,4,...,34,1,0,0,59,14,13,4,10,1
41416,0,0,1,0,0,4,0,0,0,0,...,1,0,0,0,6,0,2,9,1,1


## Especificações dos dados:

https://github.com/mauricioaniche/ck

methodAnonymousClassesQty     -->

methodAssignmentsQty          -->

methodCbo                     --> Counts the number of dependencies a class has.

methodComparisonsQty          --> The number of comparisons (i.e., == and !=).

methodLambdasQty              -->

methodLoc                     --> It counts the lines of count, ignoring empty lines and comments.

methodLoopQty                 --> The number of loops (i.e., for, while, do while, enhanced for).

methodMathOperationsQty       --> The number of math operations (times, divide, remainder, plus, minus, left and right shift).

methodMaxNestedBlocks         --> The highest number of blocks nested together.

methodNumbersQty              --> The number of numbers (i.e., int, long, double, float) literals.

methodParametersQty           -->

methodParenthesizedExpsQty    --> The number of expressions inside parenthesis.

methodReturnQty               --> The number of return instructions.

methodRfc                     --> (Response for a Class): Counts the number of unique method invocations in a class.

methodStringLiteralsQty       --> The number of string literals. Repeated strings count as many times as they appear.

methodSubClassesQty           -->

methodTryCatchQty             -->

methodUniqueWordsQty          --> counts number of unique words in the source code after removing Java keywords.

methodVariablesQty            --> Number of declared variables.

methodWmc                     --> (Weight Method Class) or McCabe's complexity.

bugFixCount                   --> Whenever any of the keywords {bug, error, mistake, fault, wrong, fail, fix} appear in the commit message, we count one more bug fix to that class.

refactoringsInvolved          --> , number of previous refactoring operations (Class).

y                             --> Output

# Random Forest Classifier

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

In [9]:
# Separação de treino e teste
X = data.drop(['y'], axis=1)
y = data['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

print(X_train.shape, X_test.shape)

(100500, 22) (49500, 22)


In [10]:
forest = RandomForestClassifier(random_state=0)
forest.fit(X_train, y_train)

RandomForestClassifier(random_state=0)

In [11]:
# Separando a lista de feature names
feature_names=list(X_train.columns)

# Explainable Technique  - SHAP

In [12]:
import shap
import joblib as jbl

In [13]:
##shap.initjs()  # Just to create better graphs =D
try:
    with open("./" + 'shap_explainer', 'rb') as f:
        explainer_shap = jbl.load(f)
except:
    explainer_shap = shap.TreeExplainer(forest)
    with open("./" + 'shap_explainer', 'wb') as f:
        jbl.dump(explainer_shap, f)

In [14]:
# Predizendo uma instância
n_row = 69
row = X_test.iloc[n_row]
to_predict = row.values.reshape(1, -1)

forest_predict = sum(forest.predict_proba(to_predict))
print("Predição do Random Forest:", forest_predict[1])

Predição do Random Forest: 0.5


/home/luana/anaconda3/envs/XAI/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [15]:
# calculate Shap values
shap_values = explainer_shap.shap_values(row)

In [16]:
shap_values = explainer_shap(row)
shap_values

.values =
array([[ 1.79032274e-05, -1.79032274e-05],
       [ 2.21933482e-02, -2.21933482e-02],
       [-9.22549512e-05,  9.22549512e-05],
       [ 3.30113483e-04, -3.30113483e-04],
       [ 6.22182306e-04, -6.22182306e-04],
       [ 8.58859124e-02, -8.58859124e-02],
       [-3.45848030e-03,  3.45848030e-03],
       [ 2.27507237e-04, -2.27507237e-04],
       [ 2.02401312e-02, -2.02401312e-02],
       [-1.88659033e-03,  1.88659033e-03],
       [-3.50887598e-02,  3.50887598e-02],
       [-7.79275409e-04,  7.79275409e-04],
       [-2.61901895e-02,  2.61901895e-02],
       [ 6.59238593e-02, -6.59238593e-02],
       [-3.67167830e-03,  3.67167830e-03],
       [-3.06774805e-05,  3.06774805e-05],
       [ 1.33893646e-04, -1.33893646e-04],
       [ 7.56560533e-03, -7.56560533e-03],
       [ 2.32483752e-02, -2.32483752e-02],
       [ 8.28977448e-03, -8.28977448e-03],
       [-9.87673442e-02,  9.87673442e-02],
       [-6.45247984e-02,  6.45247984e-02]])

.base_values =
array([[0.49981144, 0.50018

### Exportando a explicação

In [17]:
pd.DataFrame(
    data= [np.hstack((int(n_row), forest_predict[1], explainer_shap.expected_value[1], shap_values[1]))],
    columns = np.hstack(("row", "forest prediction to refactorating", "base value", list(X_test.columns)))
)

ValueError: 25 columns passed, passed data had 5 columns

In [ ]:
shap.initjs()

In [ ]:
shap.plots._waterfall.waterfall_legacy(explainer_shap.expected_value[1], shap_values[1], row)

In [ ]:
shap.force_plot(explainer_shap.expected_value[1], shap_values[1], row)

# Explainable Technique - LIME

In [ ]:
import lime
import lime.lime_tabular

In [ ]:
# LIME has one explainer for all the models
explainer_lime = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names=X_train.columns.values.tolist(), class_names=[0, 1], verbose=True, mode='classification',  discretize_continuous=True)

exp = explainer_lime.explain_instance(X_test.values[69], forest.predict_proba, num_features=8)
exp.show_in_notebook(show_table=True)

In [ ]:
exp.as_list()

# Comparando modelos de explicabilidade

In [ ]:
def export_shap_exp(row, forest_predict, base_value, shap_values):
    return pd.DataFrame(
        data = [np.hstack((row, forest_predict, base_value[1], shap_values[1]))],
        columns = np.hstack(("row", "forest prediction to refactorating", "base value", list(X_test.columns)))
    )

In [ ]:
def export_lime_exp(row, forest_predict, intercept, local_pred, lime_values):
    features = list(X_test.columns)
    exp_features = []
    exp_features_value = []
    for value in lime_values:
        # pega os nomes das features
        any((feature_name := substring) in value[0] for substring in features)
        exp_features.append(feature_name+" description")
        exp_features.append(feature_name+" weight")
        # pega os parametros de cada feature
        exp_features_value.append(value[0])
        # pega os pesos de cada feature
        exp_features_value.append(value[1])
    
    return pd.DataFrame(
        data = [np.hstack((row, forest_predict, intercept, local_pred, exp_features_value))],
        columns = np.hstack(("row", "forest prediction to refactorating", "intercept", "local prediction", exp_features))
    )

In [ ]:
def instancia_explicabilidade_local(i, shap_exps, lime_exps):
    # mostra row
    print("Instância nº: ", i)
    print(X_test.iloc[i])
    print("Predição do Random Forest: ", forest.predict_proba(X_test.iloc[[i]]))
    
    #SHAP
    print("SHAP")
    plots = []
    row = X_test.iloc[i]
    to_predict = row.values.reshape(1, -1)
    forest_predict = sum(forest.predict_proba(to_predict))
    shap_values = explainer_shap.shap_values(row)
    shap.initjs()
    display(shap.force_plot(explainer_shap.expected_value[1], shap_values[1], row))
    shap_output = export_shap_exp(i, forest_predict[1], explainer_shap.expected_value, shap_values)
    #shap_exps.join(shap_output)
    shap_result = pd.concat([shap_exps, shap_output], ignore_index=True)

    #LIME
    print("LIME")
    exp = explainer_lime.explain_instance(X_test.values[i], forest.predict_proba, num_features=8)
    exp.show_in_notebook(show_table=True)
    lime_output = export_lime_exp(i, forest_predict[1], exp.intercept[1], exp.local_pred, exp.as_list())
    #lime_exps.join(lime_output)
    lime_result = pd.concat([lime_exps, lime_output], ignore_index=True)

    print("-----------------------------------------------------------------------------------------------")

    return (shap_result, lime_result)

In [ ]:
indices_positivos = []
while len(indices_positivos) != 10:
    rand = np.random.randint(0, X_test.shape[0])
    prob = forest.predict(X_test.iloc[[rand]])
    if(prob):
        indices_positivos.append(rand)
print(indices_positivos)

In [ ]:
indices_negativos = []
while len(indices_negativos) != 10:
    rand = np.random.randint(0, X_test.shape[0])
    prob = forest.predict(X_test.iloc[[rand]])
    if(not prob):
        indices_negativos.append(rand)
print(indices_negativos)

In [ ]:
shap_output = pd.DataFrame()
lime_output = pd.DataFrame()
for ip in indices_positivos:
    (shap_output, lime_output) = instancia_explicabilidade_local(ip, shap_output, lime_output)

In [ ]:
for ineg in indices_negativos:
    (shap_output, lime_output) = instancia_explicabilidade_local(ineg, shap_output, lime_output)

In [ ]:
shap_output

In [ ]:
lime_output

In [ ]:
shap_output.to_csv('shap_explanations.csv')
lime_output.to_csv('lime_explanations.csv')

# Explainable Technique - ANCHORS

In [ ]:
from __future__ import print_function
import numpy as np
np.random.seed(1)
import sys
import sklearn
import sklearn.ensemble
from anchor import utils
from anchor import anchor_tabular

In [ ]:
explainer = anchor_tabular.AnchorTabularExplainer(
    [0, 1],
    X_train.columns.values.tolist(),
    X_train.values,
    {})

In [ ]:
X_test.iloc[[10]]

In [ ]:
idx = 10
np.random.seed(1)
print('Prediction: ', explainer.class_names[forest.predict(X_test.values[idx].reshape(1, -1))[0]])
exp = explainer.explain_instance(X_test.values[idx], forest.predict, threshold=0.95)

In [ ]:
print('Anchor: %s' % (' AND '.join(exp.names())))
print('Precision: %.2f' % exp.precision())
print('Coverage: %.2f' % exp.coverage())

In [ ]:
# Get test examples where the anchora pplies
fit_anchor = np.where(np.all(X_test.values[:, exp.features()] == X_test.values[idx][exp.features()], axis=1))[0]
print('Anchor test precision: %.2f' % (np.mean(forest.predict(X_test.values[fit_anchor]) == forest.predict(X_test.values[idx].reshape(1, -1)))))
print('Anchor test coverage: %.2f' % (fit_anchor.shape[0] / float(X_test.shape[0])))

In [ ]:
exp.show_in_notebook()

# Explainable Technique - Counterfactuals

In [ ]:
import dice_ml
from dice_ml import Dice
from dice_ml.utils import helpers  # helper functions

In [ ]:
d = dice_ml.Data(dataframe=data, continuous_features=[], outcome_name='y')
m = dice_ml.Model(model=forest, backend="sklearn")
exp = dice_ml.Dice(d,m)

In [ ]:
query = X_test.iloc[[69]].astype('int64')
query

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [ ]:
query_instance = X_test.iloc[[69]]
query_instance = query_instance.astype('int64')

e1 = exp.generate_counterfactuals(query_instance, total_CFs=10, desired_range=None,
                                  desired_class="opposite",
                                  permitted_range=None, features_to_vary="all")
e1.visualize_as_dataframe(show_only_changes=True)

In [ ]:
imp = exp.local_feature_importance(query_instance, cf_examples_list=e1.cf_examples_list)
print(imp.local_importance)

In [ ]:
imp = exp.local_feature_importance(query_instance, posthoc_sparsity_param=None)
print(imp.local_importance)

# Explainable Technique - ELi5

In [18]:
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(forest).fit(X_test, y_test)
eli5.show_weights(perm)

ImportError: cannot import name 'if_delegate_has_method' from 'sklearn.utils.metaestimators' (/home/luana/anaconda3/envs/XAI/lib/python3.12/site-packages/sklearn/utils/metaestimators.py)